In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

#Setup and Imports
# Import libraries
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

print("✅ Libraries imported successfully!")
print(f"Current time: {datetime.now()}")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 3, Finished, Available, Finished)

✅ Libraries imported successfully!
Current time: 2026-01-23 13:13:55.904835


In [ ]:
# AESO API Configuration
API_BASE_URL = "https://apimgw.aeso.ca/public/actualforecast-api/v1/load/albertaInternalLoad"

SUBSCRIPTION_KEY = "**************************************"  

# Headers
HEADERS = {
    "API-KEY": SUBSCRIPTION_KEY
}

print("✅ API configuration set!")
print(f"Base URL: {API_BASE_URL}")
print(f"Subscription key: {'*' * 20}...{SUBSCRIPTION_KEY[-4:]}") 

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 11, Finished, Available, Finished)

✅ API configuration set!
Base URL: https://apimgw.aeso.ca/public/actualforecast-api/v1/load/albertaInternalLoad
Subscription key: ********************...73c7


In [10]:
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

test_url = f"{API_BASE_URL}?startDate={start_date}&endDate={end_date}"

print(f"🔍 Testing API call...")
print(f"URL: {test_url}")
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")

# Make request
response = requests.get(test_url, headers=HEADERS)

print(f"\n📊 Response Status: {response.status_code}")

if response.status_code == 200:
    print("✅ SUCCESS! API is working!")
    
    # Parse JSON
    data = response.json()
    
    # Show structure
    print(f"\n📋 Response structure:")
    print(json.dumps(data, indent=2)[:1000])  # Show first 1000 chars
    
    # Check if we got data
    if isinstance(data, dict) and 'return' in data:
        records = data['return'].get('Actual Forecast Report', [])
        print(f"\n✅ Found {len(records)} records!")
    elif isinstance(data, list):
        print(f"\n✅ Found {len(data)} records!")
    else:
        print(f"\n⚠️ Unexpected data structure: {type(data)}")
        
else:
    print(f"❌ API call failed!")
    print(f"Status code: {response.status_code}")
    print(f"Response: {response.text[:500]}")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 12, Finished, Available, Finished)

🔍 Testing API call...
URL: https://apimgw.aeso.ca/public/actualforecast-api/v1/load/albertaInternalLoad?startDate=2026-01-16&endDate=2026-01-23
Start Date: 2026-01-16
End Date: 2026-01-23

📊 Response Status: 200
✅ SUCCESS! API is working!

📋 Response structure:
{
  "timestamp": "2026-01-23 13:26:14.701+0000",
  "responseCode": "200",
  "return": {
    "Actual Forecast Report": [
      {
        "begin_datetime_utc": "2026-01-16 07:00",
        "begin_datetime_mpt": "2026-01-16 00:00",
        "alberta_internal_load": "10394",
        "forecast_alberta_internal_load": "10390"
      },
      {
        "begin_datetime_utc": "2026-01-16 08:00",
        "begin_datetime_mpt": "2026-01-16 01:00",
        "alberta_internal_load": "10263",
        "forecast_alberta_internal_load": "10297"
      },
      {
        "begin_datetime_utc": "2026-01-16 09:00",
        "begin_datetime_mpt": "2026-01-16 02:00",
        "alberta_internal_load": "10202",
        "forecast_alberta_internal_load": "10228"


In [11]:
def fetch_aeso_load_data(start_date, end_date=None):
    """
    Fetch Alberta Internal Load data from AESO API
    
    Parameters:
    - start_date: string in format 'YYYY-MM-DD'
    - end_date: optional string in format 'YYYY-MM-DD'
    
    Returns:
    - dict: API response data
    """
    
    # Build URL
    url = f"{API_BASE_URL}?startDate={start_date}"
    if end_date:
        url += f"&endDate={end_date}"
    
    # Make request
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            print(f"✅ Fetched data for {start_date} to {end_date or 'now'}")
            return data
        else:
            print(f"❌ API Error: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Exception: {str(e)}")
        return None

# Test the function
test_data = fetch_aeso_load_data(
    start_date=(datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d"),
    end_date=datetime.now().strftime("%Y-%m-%d")
)

if test_data:
    print("\n✅ Function works!")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 13, Finished, Available, Finished)

✅ Fetched data for 2026-01-21 to 2026-01-23

✅ Function works!


In [13]:
# Fetch last 30 days of data
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

print(f"📥 Fetching AESO data from {start_date} to {end_date}...")

# Fetch data
raw_data = fetch_aeso_load_data(start_date, end_date)

if raw_data:
    # Save raw JSON to Files (Bronze - audit trail)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_filename = f"aeso_load_raw_{timestamp}.json"
    raw_path = f"/lakehouse/default/Files/bronze_raw/{raw_filename}"
    
    # Create directory if doesn't exist
    import os
    os.makedirs("/lakehouse/default/Files/bronze_raw", exist_ok=True)
    
    # Write JSON file
    with open(raw_path, 'w') as f:
        json.dump(raw_data, f, indent=2)
    
    print(f"✅ Raw JSON saved to: {raw_path}")
    
    # Parse the data structure
    # (This depends on the actual JSON structure - we'll adjust based on what we see)
    
    if isinstance(raw_data, dict) and 'return' in raw_data:
        records = raw_data['return'].get('Actual Forecast Report', [])
    elif isinstance(raw_data, list):
        records = raw_data
    else:
        print("⚠️ Need to adjust parsing logic based on actual structure")
        records = []
    
    print(f"📊 Parsed {len(records)} records")
    
    # Show sample
    if records:
        print("\n📋 Sample record:")
        print(json.dumps(records[0], indent=2))

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 15, Finished, Available, Finished)

📥 Fetching AESO data from 2025-12-24 to 2026-01-23...
✅ Fetched data for 2025-12-24 to 2026-01-23
✅ Raw JSON saved to: /lakehouse/default/Files/bronze_raw/aeso_load_raw_20260123_133006.json
📊 Parsed 744 records

📋 Sample record:
{
  "begin_datetime_utc": "2025-12-24 07:00",
  "begin_datetime_mpt": "2025-12-24 00:00",
  "alberta_internal_load": "11344",
  "forecast_alberta_internal_load": "11267"
}


In [14]:
# CELL 4: Parse AESO response and save to Bronze

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Fetch last 30 days of data
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")

print(f"📥 Fetching AESO data from {start_date} to {end_date}...")

url = f"{API_BASE_URL}?startDate={start_date}&endDate={end_date}"
response = requests.get(url, headers=HEADERS, timeout=30)

if response.status_code == 200:
    raw_data = response.json()
    
    # Extract the records array
    records = raw_data['return']['Actual Forecast Report']
    
    print(f"✅ Fetched {len(records)} hourly records")
    print(f"📅 Date range: {records[0]['begin_datetime_mpt']} to {records[-1]['begin_datetime_mpt']}")
    
    # Save raw JSON to Files (audit trail)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    raw_filename = f"aeso_load_raw_{timestamp}.json"
    
    # Create directory
    import os
    os.makedirs("/lakehouse/default/Files/bronze_raw", exist_ok=True)
    
    # Save full response
    raw_path = f"/lakehouse/default/Files/bronze_raw/{raw_filename}"
    with open(raw_path, 'w') as f:
        json.dump(raw_data, f, indent=2)
    
    print(f"✅ Raw JSON saved to: {raw_path}")
    
    # Convert to Spark DataFrame
    df_raw = spark.createDataFrame(records)
    
    # Add metadata columns
    df_bronze = df_raw \
        .withColumn("ingestion_timestamp", lit(raw_data['timestamp'])) \
        .withColumn("ingestion_date", current_date()) \
        .withColumn("api_response_code", lit(raw_data['responseCode'])) \
        .withColumn("api_start_date", lit(start_date)) \
        .withColumn("api_end_date", lit(end_date)) \
        .withColumn("raw_file_path", lit(raw_path))
    
    print("\n📋 Bronze Schema:")
    df_bronze.printSchema()
    
    print("\n📊 Sample Data:")
    df_bronze.show(5, truncate=False)
    
    # Write to Bronze Delta table
    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze_aeso_load")
    
    print(f"\n✅ {df_bronze.count()} records written to bronze_aeso_load!")
    
else:
    print(f"❌ API Error: {response.status_code}")
    print(response.text)

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 16, Finished, Available, Finished)

📥 Fetching AESO data from 2025-12-24 to 2026-01-23...
✅ Fetched 744 hourly records
📅 Date range: 2025-12-24 00:00 to 2026-01-23 23:00
✅ Raw JSON saved to: /lakehouse/default/Files/bronze_raw/aeso_load_raw_20260123_133339.json

📋 Bronze Schema:
root
 |-- alberta_internal_load: string (nullable = true)
 |-- begin_datetime_mpt: string (nullable = true)
 |-- begin_datetime_utc: string (nullable = true)
 |-- forecast_alberta_internal_load: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = false)
 |-- ingestion_date: date (nullable = false)
 |-- api_response_code: string (nullable = false)
 |-- api_start_date: string (nullable = false)
 |-- api_end_date: string (nullable = false)
 |-- raw_file_path: string (nullable = false)


📊 Sample Data:
+---------------------+------------------+------------------+------------------------------+----------------------------+--------------+-----------------+--------------+------------+----------------------------------------------------

In [16]:
# CELL 5: Verify Bronze table

# Query Bronze table
print("📊 Bronze Table Contents:")
df_bronze_check = spark.sql("""
    SELECT 
        begin_datetime_mpt,
        begin_datetime_utc,
        alberta_internal_load,
        forecast_alberta_internal_load,
        ingestion_timestamp
    FROM bronze_aeso_load 
    ORDER BY begin_datetime_utc DESC 
    LIMIT 10
""")

display(df_bronze_check)

# Get statistics
stats = spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        MIN(begin_datetime_mpt) as earliest_record,
        MAX(begin_datetime_mpt) as latest_record,
        COUNT(DISTINCT DATE(begin_datetime_mpt)) as days_of_data
    FROM bronze_aeso_load
""")

print("\n📈 Bronze Table Statistics:")
display(stats)

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 18, Finished, Available, Finished)

📊 Bronze Table Contents:


SynapseWidget(Synapse.DataFrame, e9ef9166-8428-4409-adec-a848aaf4f2eb)


📈 Bronze Table Statistics:


SynapseWidget(Synapse.DataFrame, d4e8b4a0-ecdd-4f0b-af89-298637a7374c)

In [18]:
# CELL 6: Silver Layer - Clean, type, and enrich

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Read from Bronze
df_bronze = spark.table("bronze_aeso_load")

print(f"📥 Processing {df_bronze.count()} records from Bronze...")

# Silver transformations
df_silver = df_bronze \
    .select(
        # Parse timestamps to proper timestamp types
        to_timestamp("begin_datetime_utc", "yyyy-MM-dd HH:mm").alias("datetime_utc"),
        to_timestamp("begin_datetime_mpt", "yyyy-MM-dd HH:mm").alias("datetime_mpt"),
        
        # Convert load values from string to integer
        col("alberta_internal_load").cast("integer").alias("actual_load_mw"),
        col("forecast_alberta_internal_load").cast("integer").alias("forecast_load_mw"),
        
        # Keep metadata
        col("ingestion_timestamp"),
        col("ingestion_date")
    ) \
    .filter(col("actual_load_mw").isNotNull()) \
    .filter(col("forecast_load_mw").isNotNull()) \
    .dropDuplicates(["datetime_utc"]) \
    .withColumn("forecast_error_mw", col("actual_load_mw") - col("forecast_load_mw")) \
    .withColumn("forecast_error_pct", 
                round((col("forecast_error_mw") / col("forecast_load_mw")) * 100, 2)) \
    .withColumn("absolute_error_mw", abs(col("forecast_error_mw"))) \
    .withColumn("absolute_error_pct", abs(col("forecast_error_pct"))) \
    .withColumn("hour_of_day", hour("datetime_mpt")) \
    .withColumn("day_of_week", dayofweek("datetime_mpt")) \
    .withColumn("day_name", date_format("datetime_mpt", "EEEE")) \
    .withColumn("date", date_trunc("day", "datetime_mpt")) \
    .withColumn("month", month("datetime_mpt")) \
    .withColumn("year", year("datetime_mpt")) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
    .withColumn("silver_processed_timestamp", current_timestamp())

print("\n📋 Silver Schema:")
df_silver.printSchema()

print("\n📊 Sample Silver Data:")
df_silver.select(
    "datetime_mpt",
    "actual_load_mw",
    "forecast_load_mw",
    "forecast_error_mw",
    "forecast_error_pct",
    "hour_of_day",
    "day_name"
).show(10, truncate=False)

# Write to Silver Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_aeso_load")

print(f"\n✅ {df_silver.count()} records written to silver_aeso_load!")

# Show data quality metrics
print("\n📊 Data Quality Check:")
quality_check = df_silver.select(
    count("*").alias("total_records"),
    countDistinct("datetime_utc").alias("unique_timestamps"),
    sum(when(col("actual_load_mw").isNull(), 1).otherwise(0)).alias("null_actual_load"),
    sum(when(col("forecast_load_mw").isNull(), 1).otherwise(0)).alias("null_forecast_load"),
    round(avg("absolute_error_pct"), 2).alias("avg_absolute_error_pct"),
    round(max("absolute_error_pct"), 2).alias("max_absolute_error_pct")
)

display(quality_check)

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 20, Finished, Available, Finished)

📥 Processing 744 records from Bronze...

📋 Silver Schema:
root
 |-- datetime_utc: timestamp (nullable = true)
 |-- datetime_mpt: timestamp (nullable = true)
 |-- actual_load_mw: integer (nullable = true)
 |-- forecast_load_mw: integer (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- forecast_error_mw: integer (nullable = true)
 |-- forecast_error_pct: double (nullable = true)
 |-- absolute_error_mw: integer (nullable = true)
 |-- absolute_error_pct: double (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = false)
 |-- silver_processed_timestamp: timestamp (nullable = false)


📊 Sample Silver Data:
+-------------------+--------------+----------------+-----------------+---------

SynapseWidget(Synapse.DataFrame, 498b1ce7-431c-40f5-9d92-92e40f8e8342)

In [19]:
# CELL 7: Gold Layer - Daily Load Summary

from pyspark.sql import Window

df_silver = spark.table("silver_aeso_load")

# Daily aggregations
df_daily_summary = df_silver \
    .groupBy("date", "year", "month") \
    .agg(
        count("*").alias("hourly_records"),
        min("actual_load_mw").alias("min_load_mw"),
        max("actual_load_mw").alias("max_load_mw"),
        round(avg("actual_load_mw"), 0).alias("avg_load_mw"),
        round(avg("forecast_load_mw"), 0).alias("avg_forecast_mw"),
        round(avg("absolute_error_pct"), 2).alias("avg_forecast_accuracy_pct"),
        sum("actual_load_mw").alias("total_energy_mwh")  # Sum of hourly MW = MWh
    ) \
    .withColumn("load_range_mw", col("max_load_mw") - col("min_load_mw")) \
    .withColumn("day_of_week", dayofweek("date")) \
    .withColumn("day_name", date_format("date", "EEEE")) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
    .orderBy("date")

print("📊 Daily Load Summary:")
display(df_daily_summary)

# Write to Gold
df_daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_load_summary")

print(f"\n✅ {df_daily_summary.count()} days written to gold_daily_load_summary!")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 21, Finished, Available, Finished)

📊 Daily Load Summary:


SynapseWidget(Synapse.DataFrame, 567b18c4-4e37-4ef0-bf82-5a997c3ebf0f)


✅ 31 days written to gold_daily_load_summary!


In [20]:
# CELL 8: Gold - Peak Demand by Hour

df_silver = spark.table("silver_aeso_load")

# Find peak hour for each day
window_spec = Window.partitionBy("date").orderBy(desc("actual_load_mw"))

df_peak_demand = df_silver \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .select(
        "date",
        "datetime_mpt",
        "hour_of_day",
        "day_name",
        "is_weekend",
        col("actual_load_mw").alias("peak_load_mw"),
        col("forecast_load_mw").alias("forecast_at_peak_mw"),
        "forecast_error_mw"
    ) \
    .orderBy("date")

print("📊 Peak Demand Analysis:")
display(df_peak_demand)

# Write to Gold
df_peak_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_peak_demand")

print(f"\n✅ {df_peak_demand.count()} peak records written to gold_peak_demand!")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 22, Finished, Available, Finished)

📊 Peak Demand Analysis:


SynapseWidget(Synapse.DataFrame, ef620ee5-2cee-4800-a000-024dd0bf47f4)


✅ 31 peak records written to gold_peak_demand!


In [21]:
# CELL 9: Gold - Hourly Load Patterns

df_silver = spark.table("silver_aeso_load")

# Average load by hour of day (all days combined)
df_hourly_patterns = df_silver \
    .groupBy("hour_of_day") \
    .agg(
        count("*").alias("sample_count"),
        round(avg("actual_load_mw"), 0).alias("avg_load_mw"),
        round(min("actual_load_mw"), 0).alias("min_load_mw"),
        round(max("actual_load_mw"), 0).alias("max_load_mw"),
        round(stddev("actual_load_mw"), 0).alias("stddev_load_mw")
    ) \
    .orderBy("hour_of_day")

print("📊 Average Load by Hour of Day:")
display(df_hourly_patterns)

# Weekday vs Weekend patterns
df_hourly_by_weekend = df_silver \
    .groupBy("hour_of_day", "is_weekend") \
    .agg(
        round(avg("actual_load_mw"), 0).alias("avg_load_mw"),
        count("*").alias("sample_count")
    ) \
    .orderBy("is_weekend", "hour_of_day")

print("\n📊 Weekday vs Weekend Patterns:")
display(df_hourly_by_weekend)

# Write to Gold
df_hourly_patterns.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_hourly_patterns")

df_hourly_by_weekend.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_hourly_patterns_weekend")

print("\n✅ Hourly patterns written to Gold tables!")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 23, Finished, Available, Finished)

📊 Average Load by Hour of Day:


SynapseWidget(Synapse.DataFrame, 466779b3-5ebb-4b2f-965f-6d8ab0ac9453)


📊 Weekday vs Weekend Patterns:


SynapseWidget(Synapse.DataFrame, 7ace451d-03d9-41d0-a208-aba0ec94bb13)


✅ Hourly patterns written to Gold tables!


In [22]:
# CELL 10: Gold - Forecast Accuracy Metrics

df_silver = spark.table("silver_aeso_load")

# Overall accuracy metrics
df_accuracy_overall = df_silver \
    .agg(
        count("*").alias("total_forecasts"),
        round(avg("absolute_error_pct"), 2).alias("mape_pct"),  # Mean Absolute Percentage Error
        round(avg("absolute_error_mw"), 0).alias("mae_mw"),  # Mean Absolute Error
        round(sqrt(avg(pow("forecast_error_mw", 2))), 0).alias("rmse_mw"),  # Root Mean Square Error
        round(avg("forecast_error_mw"), 0).alias("mean_bias_mw"),  # Positive = underforecast, Negative = overforecast
        min("absolute_error_pct").alias("best_forecast_error_pct"),
        max("absolute_error_pct").alias("worst_forecast_error_pct")
    )

print("📊 Overall Forecast Accuracy:")
display(df_accuracy_overall)

# Accuracy by hour of day
df_accuracy_hourly = df_silver \
    .groupBy("hour_of_day") \
    .agg(
        count("*").alias("forecast_count"),
        round(avg("absolute_error_pct"), 2).alias("mape_pct"),
        round(avg("absolute_error_mw"), 0).alias("mae_mw"),
        round(avg("forecast_error_mw"), 0).alias("mean_bias_mw")
    ) \
    .orderBy("hour_of_day")

print("\n📊 Forecast Accuracy by Hour:")
display(df_accuracy_hourly)

# Accuracy by day (trending over time)
df_accuracy_daily = df_silver \
    .groupBy("date") \
    .agg(
        count("*").alias("hourly_forecasts"),
        round(avg("absolute_error_pct"), 2).alias("daily_mape_pct"),
        round(avg("absolute_error_mw"), 0).alias("daily_mae_mw")
    ) \
    .orderBy("date")

print("\n📊 Forecast Accuracy by Day:")
display(df_accuracy_daily)

# Write to Gold
df_accuracy_overall.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_forecast_accuracy_overall")

df_accuracy_hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_forecast_accuracy_hourly")

df_accuracy_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_forecast_accuracy_daily")

print("\n✅ Forecast accuracy tables written to Gold!")

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 24, Finished, Available, Finished)

📊 Overall Forecast Accuracy:


SynapseWidget(Synapse.DataFrame, 511ede47-3733-49af-9fc0-fed2e7ba8f26)


📊 Forecast Accuracy by Hour:


SynapseWidget(Synapse.DataFrame, 4988101d-da02-4183-8dbb-a85a78511347)


📊 Forecast Accuracy by Day:


SynapseWidget(Synapse.DataFrame, ef15c417-6973-4731-b0cc-2fe052989e40)


✅ Forecast accuracy tables written to Gold!


In [23]:
# CELL 11: Summary of all tables

print("="*70)
print("🎉 AESO LOAD ANALYTICS PLATFORM - COMPLETE!")
print("="*70)

tables = [
    ("bronze_aeso_load", "Raw API responses with metadata"),
    ("silver_aeso_load", "Cleaned, typed, enriched with calculations"),
    ("gold_daily_load_summary", "Daily aggregations and statistics"),
    ("gold_peak_demand", "Peak load hour for each day"),
    ("gold_hourly_patterns", "Average load by hour of day"),
    ("gold_hourly_patterns_weekend", "Weekday vs weekend patterns"),
    ("gold_forecast_accuracy_overall", "Overall forecast metrics"),
    ("gold_forecast_accuracy_hourly", "Accuracy by hour of day"),
    ("gold_forecast_accuracy_daily", "Daily accuracy trends")
]

print("\n📊 TABLES CREATED:\n")
for table_name, description in tables:
    try:
        count = spark.table(table_name).count()
        print(f"✅ {table_name:40} {count:6} records - {description}")
    except:
        print(f"❌ {table_name:40} NOT FOUND")

print("\n" + "="*70)
print("📈 READY FOR POWER BI!")
print("="*70)

StatementMeta(, 51dbb1e9-ea2a-45f7-9f98-6779f8971cc1, 25, Finished, Available, Finished)

🎉 AESO LOAD ANALYTICS PLATFORM - COMPLETE!

📊 TABLES CREATED:

✅ bronze_aeso_load                            744 records - Raw API responses with metadata
✅ silver_aeso_load                            726 records - Cleaned, typed, enriched with calculations
✅ gold_daily_load_summary                      31 records - Daily aggregations and statistics
✅ gold_peak_demand                             31 records - Peak load hour for each day
✅ gold_hourly_patterns                         24 records - Average load by hour of day
✅ gold_hourly_patterns_weekend                 48 records - Weekday vs weekend patterns
✅ gold_forecast_accuracy_overall                1 records - Overall forecast metrics
✅ gold_forecast_accuracy_hourly                24 records - Accuracy by hour of day
✅ gold_forecast_accuracy_daily                 31 records - Daily accuracy trends

📈 READY FOR POWER BI!


In [1]:
# CELL: Check all Gold tables created
print("📊 TABLES IN YOUR LAKEHOUSE:")
print("="*60)

tables = spark.catalog.listTables()

for table in tables:
    print(f"✅ {table.name}")
    
print("\n" + "="*60)
print(f"Total tables: {len(tables)}")

StatementMeta(, 646f49f5-c785-4c37-b439-71f9e537ca14, 3, Finished, Available, Finished)

📊 TABLES IN YOUR LAKEHOUSE:
✅ bronze_aeso_load
✅ silver_aeso_load
✅ gold_daily_load_summary
✅ gold_peak_demand
✅ gold_hourly_patterns
✅ gold_hourly_patterns_weekend
✅ gold_forecast_accuracy_overall
✅ gold_forecast_accuracy_hourly
✅ gold_forecast_accuracy_daily

Total tables: 9
